# SignBridge Conversational Demo Seed (v3 - 55 Classes)

This notebook prepares a **conversational demo vocabulary** and a **demo model seed** for real-time ASL recognition.

We will:
- Load WLASL and ASL Citizen manifests (from Kaggle).
- Select a compact set of **55 glosses**: pronouns (i, you, we, he, they), core verbs, and everyday nouns.
- Build a dedicated `label_mapping_demo.json` for this vocabulary.
- Prepare a training manifest and fine-tune a **demo model** starting from the 87.6% checkpoint.

**Version History:**
- v1: 58 classes (initial)
- v2: 53 classes (removed 5 weak: food, goodbye, today, feel, me)
- v3: 55 classes (added he, they for third-person pronouns)

> **Important:** This notebook is designed to run on **Kaggle**. Paths below assume Kaggle input mounts.

In [ ]:
# ---------- Cell 1: Imports & Paths (Kaggle-style) ----------

import os
import json
from collections import Counter

import pandas as pd

# Kaggle input paths (adjust if your dataset names differ)
WLASL_INPUT = "/kaggle/input/wlasl-processed"
CITIZEN_INPUT = "/kaggle/input/asl-citizen"

# WLASL main manifest (JSON)
WLASL_JSON = os.path.join(WLASL_INPUT, "WLASL_v0.3.json")

# ASL Citizen splits directory
CITIZEN_SPLITS_DIR = os.path.join(CITIZEN_INPUT, "ASL_Citizen", "splits")

# Output directory for manifests and label mappings inside Kaggle working dir
BASE_OUTPUT_DIR = "/kaggle/working/SignBridge_demo"
MANIFESTS_DIR = os.path.join(BASE_OUTPUT_DIR, "manifests")
os.makedirs(MANIFESTS_DIR, exist_ok=True)

print("WLASL_JSON =", WLASL_JSON)
print("CITIZEN_SPLITS_DIR =", CITIZEN_SPLITS_DIR)
print("MANIFESTS_DIR =", MANIFESTS_DIR)

# Verify files exist
print("\n--- Verifying paths ---")
print(f"WLASL_JSON exists: {os.path.exists(WLASL_JSON)}")
print(f"CITIZEN_SPLITS_DIR exists: {os.path.exists(CITIZEN_SPLITS_DIR)}")

# Demo-word selection philosophy (summary):
# - Start from all glosses in WLASL and (optionally) ASL Citizen.
# - Focus on a compact set (e.g., 30–50) of:
#   * Pronouns: I, YOU, WE, THEY, etc.
#   * Core verbs: WANT, NEED, LIKE, GO, COME, HELP, KNOW, WORK, etc.
#   * Everyday nouns: FAMILY, FRIEND, SCHOOL, HOME, TIME, YEAR, etc.
# - Prefer glosses that:
#   * Are reasonably frequent in WLASL/Citizen (enough videos to train).
#   * Are already present in the existing 100-label mapping when possible,
#     to maximize reuse of the 87.6% checkpoint.
# - Result: a small, conversational vocabulary for a robust real-time demo
#   model, separate from the full 100-label general model.

In [ ]:
# ---------- Cell 2: Load WLASL Manifest & Count Instances per Gloss ----------

with open(WLASL_JSON, "r") as f:
    wlasl_data = json.load(f)

print(f"✅ Loaded WLASL manifest: {len(wlasl_data)} total glosses")

# Count instances per gloss
gloss_instance_counts = {}
for entry in wlasl_data:
    gloss = entry["gloss"].lower().strip()
    num_instances = len(entry.get("instances", []))
    gloss_instance_counts[gloss] = num_instances

# Sort by frequency (descending)
sorted_glosses = sorted(gloss_instance_counts.items(), key=lambda x: -x[1])

print(f"✅ Total unique glosses: {len(sorted_glosses)}")
print(f"✅ Total video instances: {sum(gloss_instance_counts.values())}")

# Show top 30 most frequent glosses
print("\n📊 Top 30 glosses by instance count:")
for i, (gloss, count) in enumerate(sorted_glosses[:30], 1):
    print(f"  {i:2d}. {gloss:20s} → {count} instances")

# Show some stats
counts = list(gloss_instance_counts.values())
print(f"\n📈 Instance count stats:")
print(f"   Min: {min(counts)}, Max: {max(counts)}, Median: {sorted(counts)[len(counts)//2]}")

In [ ]:
# ---------- Cell 3: Define Target Conversational Glosses & Check Availability ----------

# Our current 100-label vocabulary (from the 87.6% checkpoint)
# These are the glosses the model already knows well.
CURRENT_100_LABELS = {
    "accident", "africa", "all", "apple", "basketball", "bed", "before", "bird",
    "birthday", "black", "blue", "book", "bowling", "brown", "but", "can",
    "candy", "chair", "change", "cheat", "city", "clothes", "color", "computer",
    "cook", "cool", "corn", "cousin", "cow", "dance", "dark", "deaf", "decide",
    "doctor", "dog", "drink", "eat", "enjoy", "family", "fine", "finish", "fish",
    "forget", "full", "give", "go", "graduate", "hat", "hearing", "help", "hot",
    "how", "jacket", "kiss", "language", "last", "later", "letter", "like", "man",
    "many", "medicine", "meet", "mother", "need", "no", "now", "orange", "paint",
    "paper", "pink", "pizza", "play", "pull", "purple", "right", "same", "school",
    "secretary", "shirt", "short", "son", "study", "table", "tall", "tell",
    "thanksgiving", "thin", "thursday", "time", "walk", "want", "what", "white",
    "who", "woman", "work", "wrong", "year", "yes"
}

print(f"✅ Current 100-label vocabulary loaded: {len(CURRENT_100_LABELS)} glosses")

# Target conversational glosses we WANT for the demo
# These are words that make natural short sentences possible.
TARGET_CONVERSATIONAL = {
    # Pronouns (critical for sentences!)
    "i", "you", "we", "they", "he", "she", "it", "my", "your", "our", "me",
    
    # Core verbs (actions)
    "want", "need", "like", "go", "come", "help", "know", "have", "see", "hear",
    "think", "feel", "love", "hate", "make", "take", "give", "get", "put", "use",
    "work", "play", "eat", "drink", "sleep", "wake", "run", "walk", "sit", "stand",
    "read", "write", "learn", "teach", "study", "understand", "remember", "forget",
    "try", "start", "stop", "finish", "wait", "ask", "tell", "say", "talk", "call",
    "meet", "visit", "live", "stay", "leave", "arrive", "move", "buy", "sell", "pay",
    
    # Question words
    "what", "who", "where", "when", "why", "how", "which",
    
    # Common nouns (people, places, things)
    "family", "friend", "mother", "father", "brother", "sister", "son", "daughter",
    "baby", "child", "man", "woman", "person", "people", "name",
    "home", "house", "school", "work", "office", "store", "hospital", "city",
    "food", "water", "money", "time", "day", "night", "morning", "afternoon", "week", "month", "year",
    "today", "tomorrow", "yesterday", "now", "later", "before", "after",
    
    # Adjectives / states
    "good", "bad", "happy", "sad", "hungry", "tired", "sick", "fine", "busy", "free",
    "new", "old", "big", "small", "hot", "cold", "easy", "hard",
    
    # Basic responses / connectors
    "yes", "no", "please", "thank", "sorry", "hello", "goodbye", "ok", "more", "again",
}

print(f"✅ Target conversational glosses: {len(TARGET_CONVERSATIONAL)} words")

# Check which target glosses exist in WLASL
available_in_wlasl = {}
missing_from_wlasl = []

for gloss in TARGET_CONVERSATIONAL:
    if gloss in gloss_instance_counts:
        available_in_wlasl[gloss] = gloss_instance_counts[gloss]
    else:
        missing_from_wlasl.append(gloss)

print(f"\n📊 Target glosses available in WLASL: {len(available_in_wlasl)}/{len(TARGET_CONVERSATIONAL)}")
print(f"❌ Target glosses NOT in WLASL: {len(missing_from_wlasl)}")

# Categorize available glosses
already_in_100 = {g: c for g, c in available_in_wlasl.items() if g in CURRENT_100_LABELS}
new_glosses = {g: c for g, c in available_in_wlasl.items() if g not in CURRENT_100_LABELS}

print(f"\n✅ Already in our 100-label model: {len(already_in_100)} glosses")
print(f"🆕 NEW glosses to add (exist in WLASL, not in our 100): {len(new_glosses)} glosses")

# Show the new glosses with their instance counts
print("\n🆕 NEW conversational glosses available in WLASL:")
for gloss, count in sorted(new_glosses.items(), key=lambda x: -x[1]):
    print(f"   {gloss:15s} → {count:2d} instances")

# Show what's missing entirely from WLASL
if missing_from_wlasl:
    print(f"\n❌ Target glosses not found in WLASL (cannot add):")
    print(f"   {', '.join(sorted(missing_from_wlasl))}")

In [ ]:
# ---------- Cell 4: Build Final Demo Vocabulary ----------

# Minimum instances required to include a gloss (for trainability)
MIN_INSTANCES = 7

# Filter new glosses by minimum instance count
new_glosses_filtered = {g: c for g, c in new_glosses.items() if c >= MIN_INSTANCES}
print(f"🆕 New glosses with >= {MIN_INSTANCES} instances: {len(new_glosses_filtered)}")

# From the existing 100-label set, select the ones that are useful for conversation
# (We already checked these are in CURRENT_100_LABELS)
USEFUL_FROM_100 = {
    # Verbs already in our model
    "can", "change", "cook", "dance", "decide", "drink", "eat", "enjoy",
    "finish", "forget", "give", "go", "help", "like", "meet", "need",
    "paint", "play", "pull", "study", "tell", "walk", "want", "work",
    
    # Nouns already in our model
    "family", "mother", "son", "man", "woman", "doctor", "secretary",
    "school", "city", "book", "letter", "paper", "medicine", "pizza",
    "dog", "bird", "fish", "cow",
    
    # Question words / time / connectors
    "what", "who", "how", "now", "later", "before", "last", "time", "year",
    
    # Adjectives / states
    "fine", "hot", "cool", "dark", "full", "short", "tall", "thin",
    "right", "wrong", "same", "deaf", "hearing",
    
    # Responses
    "yes", "no",
}

# Verify all USEFUL_FROM_100 are actually in CURRENT_100_LABELS
assert USEFUL_FROM_100.issubset(CURRENT_100_LABELS), "Some glosses not in 100-label set!"
print(f"✅ Useful glosses from existing 100-label model: {len(USEFUL_FROM_100)}")

# Combine: existing useful + all new filtered
demo_vocab = USEFUL_FROM_100.union(set(new_glosses_filtered.keys()))
print(f"\n🎯 FINAL DEMO VOCABULARY: {len(demo_vocab)} glosses")

# Categorize for display
demo_from_100 = demo_vocab.intersection(CURRENT_100_LABELS)
demo_new = demo_vocab - CURRENT_100_LABELS

print(f"   • From existing 100-label model: {len(demo_from_100)}")
print(f"   • NEW glosses (not in 100): {len(demo_new)}")

# Show the NEW glosses that will be added (sorted by instance count)
print(f"\n🆕 NEW glosses to add ({len(demo_new)} total):")
new_sorted = sorted([(g, new_glosses[g]) for g in demo_new], key=lambda x: -x[1])
for i, (gloss, count) in enumerate(new_sorted, 1):
    print(f"   {i:3d}. {gloss:15s} → {count:2d} instances")

# Show the glosses we're keeping from the 100-label model
print(f"\n✅ Keeping from existing model ({len(demo_from_100)} total):")
print(f"   {', '.join(sorted(demo_from_100))}")

In [ ]:
# ---------- Cell 5: Compact Dialogue-Focused Demo Vocabulary (v3 - 55 classes) ----------

# Let's define actual dialogue scenarios we want to demo, then pick ONLY the glosses needed.

# === DEMO DIALOGUE SCENARIOS ===
# 
# Dialogue 1: Greeting & How are you
#   A: "HELLO, HOW YOU?"
#   B: "I FINE, YOU?"
#   A: "I GOOD"
#
# Dialogue 2: Asking about plans / needs
#   A: "YOU WORK TODAY?"
#   B: "NO, I STAY HOME"
#   A: "YOU NEED HELP?"
#   B: "YES PLEASE"
#
# Dialogue 3: Talking about family / feelings
#   A: "HOW FAMILY?"
#   B: "MOTHER FINE, FATHER SICK"
#   A: "SORRY, I HOPE BETTER"
#   B: "YES, SOON"
#
# Dialogue 4: Food / daily activities
#   A: "YOU HUNGRY?"
#   B: "YES, I WANT EAT"
#   A: "WHAT YOU LIKE?"
#   B: "I LIKE PIZZA"
#
# Dialogue 5: Talking about others
#   A: "WHERE HE GO?"
#   B: "HE WORK"
#   A: "THEY COME LATER?"
#   B: "YES, THEY COME"

# === COMPACT DEMO VOCABULARY v3 (55 classes) ===
# v1: 58 classes
# v2: 53 classes (removed 5 weak: food, goodbye, today, feel, me)
# v3: 55 classes (added he, they for third-person pronouns)

DEMO_VOCAB_COMPACT = {
    # === PRONOUNS (essential for any sentence) ===
    "i", "you", "my", "your", "we", "he", "they",  # Added: he (30), they (67)
    
    # === CORE VERBS ===
    "want", "need", "like", "go", "help", "know", "have",
    "work", "eat", "drink", "stay", "see",
    
    # === QUESTION WORDS ===
    "what", "who", "how", "where", "when", "why",
    
    # === PEOPLE / FAMILY ===
    "family", "mother", "father", "friend", "man", "woman",
    
    # === PLACES / THINGS ===
    "home", "school", "water", "pizza", "time", "day",
    
    # === ADJECTIVES / STATES ===
    "good", "fine", "happy", "tired", "hungry", "sick", "hot", "cold",
    
    # === TIME WORDS ===
    "tomorrow", "now", "later", "before",
    
    # === RESPONSES / CONNECTORS ===
    "yes", "no", "please", "sorry", "hello", "ok",
}

print(f"🎯 COMPACT DEMO VOCABULARY v3: {len(DEMO_VOCAB_COMPACT)} glosses")
print(f"   Added: he (30 samples), they (67 samples)")
print(f"   Skipped: she (9), it (0) - insufficient samples")

# Check availability in WLASL
available = {g for g in DEMO_VOCAB_COMPACT if g in gloss_instance_counts or g in citizen_gloss_counts}
missing = DEMO_VOCAB_COMPACT - available

print(f"\n   ✅ Available in WLASL/Citizen: {len(available)}")
print(f"   ❌ Missing from both: {len(missing)}")
if missing:
    print(f"      Missing: {', '.join(sorted(missing))}")

# Check which are already in our 100-label model vs new
from_100 = available.intersection(CURRENT_100_LABELS)
new_to_add = available - CURRENT_100_LABELS

print(f"\n📊 Breakdown:")
print(f"   • Already in 100-label model (reuse weights): {len(from_100)}")
print(f"   • NEW glosses to learn: {len(new_to_add)}")

# Show instance counts for all demo glosses (WLASL + Citizen combined)
print(f"\n📋 Demo vocabulary with instance counts:")
demo_with_counts = []
for g in sorted(available):
    wlasl = gloss_instance_counts.get(g, 0)
    citizen = citizen_gloss_counts.get(g, 0)
    total = wlasl + citizen
    demo_with_counts.append((g, total))

for gloss, count in sorted(demo_with_counts, key=lambda x: -x[1]):
    source = "✓ in 100" if gloss in CURRENT_100_LABELS else "🆕 NEW"
    print(f"   {gloss:12s} → {count:3d} instances  [{source}]")

In [ ]:
# ---------- Cell 6: Cross-Reference with ASL Citizen for More Samples ----------

from collections import defaultdict

# Load Citizen splits and count instances per gloss
citizen_gloss_counts = defaultdict(int)
citizen_split_counts = defaultdict(lambda: defaultdict(int))  # split -> gloss -> count

for split_file in ["train.csv", "val.csv", "test.csv"]:
    split_path = os.path.join(CITIZEN_SPLITS_DIR, split_file)
    if not os.path.exists(split_path):
        print(f"⚠️ {split_file} not found at {split_path}")
        continue
    
    df = pd.read_csv(split_path)
    split_name = split_file.replace(".csv", "")
    
    # Find the gloss column (usually 'Gloss' or first column)
    gloss_col = "Gloss" if "Gloss" in df.columns else df.columns[0]
    
    for gloss_raw in df[gloss_col]:
        # Normalize: "APPLE" -> "apple", "SOCCER2" -> "soccer"
        gloss = ''.join([c for c in str(gloss_raw) if not c.isdigit()]).strip().lower()
        citizen_gloss_counts[gloss] += 1
        citizen_split_counts[split_name][gloss] += 1
    
    print(f"✅ Loaded {split_file}: {len(df)} rows")

print(f"\n✅ Total unique glosses in Citizen: {len(citizen_gloss_counts)}")
print(f"✅ Total Citizen videos: {sum(citizen_gloss_counts.values())}")

# Cross-reference with our compact demo vocab
print(f"\n📊 Demo glosses in ASL Citizen:")
demo_in_citizen = {}
demo_not_in_citizen = []

for gloss in sorted(available):  # 'available' from Cell 5 = our 58 demo glosses
    if gloss in citizen_gloss_counts:
        demo_in_citizen[gloss] = citizen_gloss_counts[gloss]
    else:
        demo_not_in_citizen.append(gloss)

print(f"   ✅ Found in Citizen: {len(demo_in_citizen)}/{len(available)}")
print(f"   ❌ Not in Citizen: {len(demo_not_in_citizen)}")

if demo_not_in_citizen:
    print(f"      Missing: {', '.join(demo_not_in_citizen)}")

# Show combined totals (WLASL + Citizen)
print(f"\n📋 Demo vocabulary: WLASL + Citizen combined")
print(f"{'Gloss':<12} {'WLASL':>6} {'Citizen':>8} {'TOTAL':>7}  Source")
print("-" * 50)

combined_data = []
for gloss in sorted(available):
    wlasl_count = gloss_instance_counts.get(gloss, 0)
    citizen_count = citizen_gloss_counts.get(gloss, 0)
    total = wlasl_count + citizen_count
    source = "✓ in 100" if gloss in CURRENT_100_LABELS else "🆕 NEW"
    combined_data.append((gloss, wlasl_count, citizen_count, total, source))

# Sort by total count descending
for gloss, wlasl, citizen, total, source in sorted(combined_data, key=lambda x: -x[3]):
    print(f"{gloss:<12} {wlasl:>6} {citizen:>8} {total:>7}  [{source}]")

# Summary stats
total_wlasl = sum(x[1] for x in combined_data)
total_citizen = sum(x[2] for x in combined_data)
print(f"\n📈 TOTALS: WLASL={total_wlasl}, Citizen={total_citizen}, Combined={total_wlasl + total_citizen}")

In [ ]:
# ---------- Cell 7: Finalize Demo Vocab & Create Label Mapping ----------

# Low-count glosses (< 15 combined instances) - may need extra augmentation
LOW_COUNT_THRESHOLD = 15
low_count_glosses = [(g, w, c, w+c) for g, w, c, t, s in combined_data if w + c < LOW_COUNT_THRESHOLD]

if low_count_glosses:
    print(f"⚠️ Low-count glosses (< {LOW_COUNT_THRESHOLD} combined instances):")
    for gloss, wlasl, citizen, total in low_count_glosses:
        print(f"   {gloss}: {total} instances (will need heavy augmentation)")
else:
    print(f"✅ All glosses have >= {LOW_COUNT_THRESHOLD} combined instances")

# Final demo vocabulary (keeping all 58 - we'll rely on augmentation for low-count ones)
FINAL_DEMO_VOCAB = sorted(available)  # alphabetically sorted for consistent label assignment

print(f"\n🎯 FINAL DEMO VOCABULARY: {len(FINAL_DEMO_VOCAB)} glosses")

# Create label mapping
gloss_to_label_demo = {gloss: idx for idx, gloss in enumerate(FINAL_DEMO_VOCAB)}
label_to_gloss_demo = {idx: gloss for gloss, idx in gloss_to_label_demo.items()}

label_mapping_demo = {
    "gloss_to_label": gloss_to_label_demo,
    "label_to_gloss": label_to_gloss_demo,
    "num_classes": len(FINAL_DEMO_VOCAB),
    "description": "Compact conversational demo vocabulary for SignBridge real-time demo",
    "source": "WLASL + ASL Citizen intersection, dialogue-focused selection"
}

# Save to manifests directory
label_mapping_path = os.path.join(MANIFESTS_DIR, "label_mapping_demo.json")
with open(label_mapping_path, "w") as f:
    json.dump(label_mapping_demo, f, indent=2)

print(f"✅ Saved label mapping to: {label_mapping_path}")

# Show the mapping
print(f"\n📋 Label Mapping (gloss → label):")
for gloss, label in gloss_to_label_demo.items():
    source = "✓ in 100" if gloss in CURRENT_100_LABELS else "🆕 NEW"
    print(f"   {label:2d} → {gloss:12s}  [{source}]")

In [ ]:
# ---------- Cell 8: Build Training Manifest from WLASL + Citizen (with existence check) ----------

# We'll create a combined manifest CSV with columns:
#   video_id, video_path, gloss, label, split, source

# ========== Build set of existing WLASL video IDs ==========
WLASL_VIDEOS_DIR = os.path.join(WLASL_INPUT, "videos")

print(f"🔍 Scanning WLASL videos directory: {WLASL_VIDEOS_DIR}")
existing_wlasl_ids = set()
for f in os.listdir(WLASL_VIDEOS_DIR):
    if f.endswith(".mp4"):
        existing_wlasl_ids.add(f.replace(".mp4", ""))

print(f"✅ Found {len(existing_wlasl_ids)} existing WLASL video files")

# ========== WLASL Videos (with existence check) ==========
wlasl_records = []
wlasl_skipped = 0

for entry in wlasl_data:
    gloss = entry["gloss"].lower().strip()
    if gloss not in gloss_to_label_demo:
        continue  # skip glosses not in our demo vocab
    
    label = gloss_to_label_demo[gloss]
    
    for instance in entry.get("instances", []):
        video_id = instance.get("video_id", "")
        split = instance.get("split", "train")
        
        # Check if video actually exists
        if video_id not in existing_wlasl_ids:
            wlasl_skipped += 1
            continue
        
        video_path = os.path.join(WLASL_VIDEOS_DIR, f"{video_id}.mp4")
        
        wlasl_records.append({
            "video_id": video_id,
            "video_path": video_path,
            "gloss": gloss,
            "label": label,
            "split": split,
            "source": "wlasl"
        })

print(f"✅ WLASL records for demo vocab: {len(wlasl_records)} (skipped {wlasl_skipped} missing videos)")

# ========== Citizen Videos ==========
CITIZEN_VIDEOS_DIR = os.path.join(CITIZEN_INPUT, "ASL_Citizen", "videos")

citizen_records = []
for split_file in ["train.csv", "val.csv", "test.csv"]:
    split_path = os.path.join(CITIZEN_SPLITS_DIR, split_file)
    if not os.path.exists(split_path):
        continue
    
    df = pd.read_csv(split_path)
    split_name = split_file.replace(".csv", "")
    
    # Find columns
    gloss_col = "Gloss" if "Gloss" in df.columns else df.columns[0]
    video_col = None
    for candidate in ["video_id", "Video", "filename", "file", "video"]:
        if candidate in df.columns:
            video_col = candidate
            break
    if video_col is None:
        video_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
    
    for _, row in df.iterrows():
        gloss_raw = str(row[gloss_col])
        gloss = ''.join([c for c in gloss_raw if not c.isdigit()]).strip().lower()
        
        if gloss not in gloss_to_label_demo:
            continue
        
        label = gloss_to_label_demo[gloss]
        video_id = str(row[video_col]) if video_col else ""
        
        if not video_id.endswith(".mp4"):
            video_filename = f"{video_id}.mp4"
        else:
            video_filename = video_id
        
        video_path = os.path.join(CITIZEN_VIDEOS_DIR, video_filename)
        
        citizen_records.append({
            "video_id": video_id,
            "video_path": video_path,
            "gloss": gloss,
            "label": label,
            "split": split_name,
            "source": "citizen"
        })

print(f"✅ Citizen records for demo vocab: {len(citizen_records)}")

# ========== Combine ==========
all_records = wlasl_records + citizen_records
df_manifest = pd.DataFrame(all_records)

print(f"\n📊 Combined manifest:")
print(f"   Total records: {len(df_manifest)}")
print(f"   By source: {df_manifest['source'].value_counts().to_dict()}")
print(f"   By split: {df_manifest['split'].value_counts().to_dict()}")

# Save manifest
manifest_path = os.path.join(MANIFESTS_DIR, "demo_training_manifest.csv")
df_manifest.to_csv(manifest_path, index=False)
print(f"\n✅ Saved manifest to: {manifest_path}")

# Show sample
print(f"\n📋 Sample records:")
print(df_manifest.head(10).to_string(index=False))

In [ ]:
# ---------- Cell 9: Re-split manifest for fine-tuning (train + test as train) ----------

# For this demo-focused fine-tuning, we care more about using as much data as possible
# than about having a separate held-out *test* set.
#
# Strategy:
#   - Use all rows with split in {"train", "test"} as TRAIN.
#   - Keep existing "val" rows as VALIDATION for early stopping / sanity checks.

manifest_path = os.path.join(MANIFESTS_DIR, "demo_training_manifest.csv")
df = pd.read_csv(manifest_path)

print("📥 Loaded manifest:")
print(df["split"].value_counts())

train_mask = df["split"].isin(["train", "test"])
val_mask = df["split"] == "val"

train_df = df[train_mask].copy()
val_df = df[val_mask].copy()

# Normalize split names
train_df["split"] = "train"
val_df["split"] = "val"

print("\n📊 New split sizes:")
print(f"   TRAIN: {len(train_df)} (was train+test)")
print(f"   VAL:   {len(val_df)} (original val)")

# Save separate manifests
train_manifest_path = os.path.join(MANIFESTS_DIR, "demo_train_manifest.csv")
val_manifest_path = os.path.join(MANIFESTS_DIR, "demo_val_manifest.csv")

train_df.to_csv(train_manifest_path, index=False)
val_df.to_csv(val_manifest_path, index=False)

print(f"\n✅ Saved TRAIN manifest to: {train_manifest_path}")
print(f"✅ Saved VAL manifest to:   {val_manifest_path}")

print("\n📋 TRAIN sample:")
print(train_df.head(10).to_string(index=False))

print("\n📋 VAL sample:")
print(val_df.head(10).to_string(index=False))

In [ ]:
# ---------- Cell 10: Load 87.6% Checkpoint & Prepare Weight Transfer ----------

import torch
import torch.nn as nn

# Load the 87.6% checkpoint (100 classes) from Kaggle input
CHECKPOINT_100_PATH = "/kaggle/input/best-model-87/best_model_citizen100_87pct.pth"

# Load the original 100-class label mapping (for weight transfer)
LABEL_MAPPING_100_PATH = "/kaggle/input/label-map-100/label_mapping.json"

print(f"📦 Loading 100-class checkpoint from: {CHECKPOINT_100_PATH}")
checkpoint_100 = torch.load(CHECKPOINT_100_PATH, map_location="cpu")

# Check what's in the checkpoint
if isinstance(checkpoint_100, dict):
    print(f"   Keys: {list(checkpoint_100.keys())}")
    if "model_state_dict" in checkpoint_100:
        state_dict_100 = checkpoint_100["model_state_dict"]
    else:
        state_dict_100 = checkpoint_100
else:
    state_dict_100 = checkpoint_100

print(f"   State dict has {len(state_dict_100)} keys")

with open(LABEL_MAPPING_100_PATH, "r") as f:
    label_mapping_100 = json.load(f)

gloss_to_label_100 = label_mapping_100["gloss_to_label"]
print(f"✅ Loaded 100-class label mapping: {len(gloss_to_label_100)} glosses")

# Our demo N-class mapping (from Cell 7)
NUM_CLASSES_DEMO = len(gloss_to_label_demo)
print(f"✅ Demo {NUM_CLASSES_DEMO}-class label mapping: {NUM_CLASSES_DEMO} glosses")

# Find overlapping glosses (present in both 100-class and demo)
overlapping_glosses = set(gloss_to_label_100.keys()).intersection(set(gloss_to_label_demo.keys()))
print(f"\n🔗 Overlapping glosses (can transfer weights): {len(overlapping_glosses)}")

# Build mapping: demo_label -> 100_label (for weight copying)
demo_to_100_label = {}
for gloss in overlapping_glosses:
    demo_label = gloss_to_label_demo[gloss]
    old_label = gloss_to_label_100[gloss]
    demo_to_100_label[demo_label] = old_label

print(f"   Will copy classifier weights for {len(demo_to_100_label)} classes")

In [ ]:
# ---------- Cell 11: Build Demo I3D Model with Weight Transfer ----------

import torch.nn.functional as F

# ============================================================================
# I3D Model Definition (matching CV/models/i3d.py from SignBridge repo)
# ============================================================================

class MaxPool3dSamePadding(nn.MaxPool3d):
    def compute_pad(self, dim, s):
        if s % self.stride[dim] == 0:
            return max(self.kernel_size[dim] - self.stride[dim], 0)
        else:
            return max(self.kernel_size[dim] - (s % self.stride[dim]), 0)

    def forward(self, x):
        batch, channel, t, h, w = x.size()
        pad_t = self.compute_pad(0, t)
        pad_h = self.compute_pad(1, h)
        pad_w = self.compute_pad(2, w)
        pad_t_f, pad_t_b = pad_t // 2, pad_t - pad_t // 2
        pad_h_f, pad_h_b = pad_h // 2, pad_h - pad_h // 2
        pad_w_f, pad_w_b = pad_w // 2, pad_w - pad_w // 2
        x = F.pad(x, (pad_w_f, pad_w_b, pad_h_f, pad_h_b, pad_t_f, pad_t_b))
        return super().forward(x)


class Unit3D(nn.Module):
    def __init__(self, in_channels, output_channels, kernel_shape=(1,1,1),
                 stride=(1,1,1), padding=0, activation_fn=F.relu,
                 use_batch_norm=True, use_bias=False, name="unit_3d"):
        super().__init__()
        self._output_channels = output_channels
        self._kernel_shape = kernel_shape
        self._stride = stride
        self._use_batch_norm = use_batch_norm
        self._activation_fn = activation_fn
        self._use_bias = use_bias
        self.name = name
        self.conv3d = nn.Conv3d(in_channels, output_channels, kernel_shape, stride, padding=0, bias=use_bias)
        if use_batch_norm:
            self.bn = nn.BatchNorm3d(output_channels, eps=0.001, momentum=0.01)

    def compute_pad(self, dim, s):
        if s % self._stride[dim] == 0:
            return max(self._kernel_shape[dim] - self._stride[dim], 0)
        else:
            return max(self._kernel_shape[dim] - (s % self._stride[dim]), 0)

    def forward(self, x):
        batch, channel, t, h, w = x.size()
        pad_t = self.compute_pad(0, t)
        pad_h = self.compute_pad(1, h)
        pad_w = self.compute_pad(2, w)
        pad_t_f, pad_t_b = pad_t // 2, pad_t - pad_t // 2
        pad_h_f, pad_h_b = pad_h // 2, pad_h - pad_h // 2
        pad_w_f, pad_w_b = pad_w // 2, pad_w - pad_w // 2
        x = F.pad(x, (pad_w_f, pad_w_b, pad_h_f, pad_h_b, pad_t_f, pad_t_b))
        x = self.conv3d(x)
        if self._use_batch_norm:
            x = self.bn(x)
        if self._activation_fn is not None:
            x = self._activation_fn(x)
        return x


class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_channels, name):
        super().__init__()
        self.b0 = Unit3D(in_channels, out_channels[0], kernel_shape=[1,1,1], name=name+"/Branch_0/Conv3d_0a_1x1")
        self.b1a = Unit3D(in_channels, out_channels[1], kernel_shape=[1,1,1], name=name+"/Branch_1/Conv3d_0a_1x1")
        self.b1b = Unit3D(out_channels[1], out_channels[2], kernel_shape=[3,3,3], name=name+"/Branch_1/Conv3d_0b_3x3")
        self.b2a = Unit3D(in_channels, out_channels[3], kernel_shape=[1,1,1], name=name+"/Branch_2/Conv3d_0a_1x1")
        self.b2b = Unit3D(out_channels[3], out_channels[4], kernel_shape=[3,3,3], name=name+"/Branch_2/Conv3d_0b_3x3")
        self.b3a = MaxPool3dSamePadding(kernel_size=[3,3,3], stride=(1,1,1))
        self.b3b = Unit3D(in_channels, out_channels[5], kernel_shape=[1,1,1], name=name+"/Branch_3/Conv3d_0b_1x1")

    def forward(self, x):
        b0 = self.b0(x)
        b1 = self.b1b(self.b1a(x))
        b2 = self.b2b(self.b2a(x))
        b3 = self.b3b(self.b3a(x))
        return torch.cat([b0, b1, b2, b3], dim=1)


class InceptionI3d(nn.Module):
    def __init__(self, num_classes=400, spatial_squeeze=True, in_channels=3, dropout_keep_prob=0.5):
        super().__init__()
        self.num_classes = num_classes
        self.spatial_squeeze = spatial_squeeze
        self.Conv3d_1a_7x7 = Unit3D(in_channels, 64, kernel_shape=[7,7,7], stride=(2,2,2), name="Conv3d_1a_7x7")
        self.MaxPool3d_2a_3x3 = MaxPool3dSamePadding(kernel_size=[1,3,3], stride=(1,2,2))
        self.Conv3d_2b_1x1 = Unit3D(64, 64, kernel_shape=[1,1,1], name="Conv3d_2b_1x1")
        self.Conv3d_2c_3x3 = Unit3D(64, 192, kernel_shape=[3,3,3], name="Conv3d_2c_3x3")
        self.MaxPool3d_3a_3x3 = MaxPool3dSamePadding(kernel_size=[1,3,3], stride=(1,2,2))
        self.Mixed_3b = InceptionModule(192, [64,96,128,16,32,32], "Mixed_3b")
        self.Mixed_3c = InceptionModule(256, [128,128,192,32,96,64], "Mixed_3c")
        self.MaxPool3d_4a_3x3 = MaxPool3dSamePadding(kernel_size=[3,3,3], stride=(2,2,2))
        self.Mixed_4b = InceptionModule(480, [192,96,208,16,48,64], "Mixed_4b")
        self.Mixed_4c = InceptionModule(512, [160,112,224,24,64,64], "Mixed_4c")
        self.Mixed_4d = InceptionModule(512, [128,128,256,24,64,64], "Mixed_4d")
        self.Mixed_4e = InceptionModule(512, [112,144,288,32,64,64], "Mixed_4e")
        self.Mixed_4f = InceptionModule(528, [256,160,320,32,128,128], "Mixed_4f")
        self.MaxPool3d_5a_2x2 = MaxPool3dSamePadding(kernel_size=[2,2,2], stride=(2,2,2))
        self.Mixed_5b = InceptionModule(832, [256,160,320,32,128,128], "Mixed_5b")
        self.Mixed_5c = InceptionModule(832, [384,192,384,48,128,128], "Mixed_5c")
        self.avg_pool = nn.AvgPool3d(kernel_size=[2,7,7], stride=(1,1,1))
        self.dropout = nn.Dropout(dropout_keep_prob)
        self.logits = Unit3D(1024, num_classes, kernel_shape=[1,1,1], activation_fn=None,
                             use_batch_norm=False, use_bias=True, name="logits")

    def replace_logits(self, num_classes):
        self.num_classes = num_classes
        self.logits = Unit3D(1024, num_classes, kernel_shape=[1,1,1], activation_fn=None,
                             use_batch_norm=False, use_bias=True, name="logits")

    def forward(self, x):
        x = self.Conv3d_1a_7x7(x)
        x = self.MaxPool3d_2a_3x3(x)
        x = self.Conv3d_2b_1x1(x)
        x = self.Conv3d_2c_3x3(x)
        x = self.MaxPool3d_3a_3x3(x)
        x = self.Mixed_3b(x)
        x = self.Mixed_3c(x)
        x = self.MaxPool3d_4a_3x3(x)
        x = self.Mixed_4b(x)
        x = self.Mixed_4c(x)
        x = self.Mixed_4d(x)
        x = self.Mixed_4e(x)
        x = self.Mixed_4f(x)
        x = self.MaxPool3d_5a_2x2(x)
        x = self.Mixed_5b(x)
        x = self.Mixed_5c(x)
        x = self.avg_pool(x)
        x = self.dropout(x)
        x = self.logits(x)
        if self.spatial_squeeze:
            x = x.squeeze(3).squeeze(3)
        x = x.mean(2)
        return x


# ============================================================================
# Create demo model and transfer weights
# ============================================================================

print(f"🎯 Building {NUM_CLASSES_DEMO}-class demo model...")

# Step 1: Create model with 100 classes first to load checkpoint
model_100 = InceptionI3d(num_classes=100, in_channels=3, dropout_keep_prob=0.5)
model_100.load_state_dict(state_dict_100)
print("✅ Loaded 100-class weights into temporary model")

# Step 2: Create the demo model with correct number of classes
model_demo = InceptionI3d(num_classes=NUM_CLASSES_DEMO, in_channels=3, dropout_keep_prob=0.5)

# Step 3: Copy backbone weights (everything except logits)
backbone_state = {k: v for k, v in model_100.state_dict().items() if not k.startswith("logits")}
model_demo.load_state_dict(backbone_state, strict=False)
print(f"✅ Copied backbone weights ({len(backbone_state)} keys)")

# Step 4: Transfer classifier weights for overlapping classes
old_weight = model_100.logits.conv3d.weight.data  # [100, 1024, 1, 1, 1]
old_bias = model_100.logits.conv3d.bias.data      # [100]

new_weight = model_demo.logits.conv3d.weight.data  # [NUM_CLASSES_DEMO, 1024, 1, 1, 1]
new_bias = model_demo.logits.conv3d.bias.data      # [NUM_CLASSES_DEMO]

transferred = 0
for demo_label, old_label in demo_to_100_label.items():
    new_weight[demo_label] = old_weight[old_label]
    new_bias[demo_label] = old_bias[old_label]
    transferred += 1

print(f"✅ Transferred classifier weights for {transferred} overlapping classes")
print(f"🆕 Remaining {NUM_CLASSES_DEMO - transferred} classes initialized randomly")

# Clean up
del model_100
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n🎯 Demo model ready: InceptionI3d with {NUM_CLASSES_DEMO} classes")
print(f"   Total parameters: {sum(p.numel() for p in model_demo.parameters()):,}")

## ⚠️ SWITCH TO CPU FOR PREPROCESSING

The next cell preprocesses all 2941 demo videos to `.npz` format. This is CPU-bound (video decoding + resizing), so:

1. **Use CPU accelerator** (saves GPU quota)
2. **Uses 4 workers** (parallel processing)
3. Takes ~10-15 minutes for 2941 videos

After preprocessing completes, **switch to GPU** for training.

In [ ]:
# ---------- Cell 13: Preprocess Demo Videos to NPZ (CPU, 4 workers) ----------

import numpy as np
import cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Preprocessing config (matching your 87.6% training)
NUM_FRAMES = 32
IMAGE_SIZE = 224

# Output directory for preprocessed files
PREPROCESSED_DIR = os.path.join(BASE_OUTPUT_DIR, "preprocessed")
os.makedirs(os.path.join(PREPROCESSED_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(PREPROCESSED_DIR, "val"), exist_ok=True)

print(f"📁 Preprocessed output: {PREPROCESSED_DIR}")

def preprocess_video(video_path: str, num_frames: int = 32, image_size: int = 224):
    """
    Read video, sample frames uniformly, resize to (image_size, image_size).
    Returns: numpy array of shape (num_frames, H, W, 3), dtype uint8
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    
    if len(frames) == 0:
        return None
    
    # Sample frames uniformly to get exactly num_frames
    total = len(frames)
    if total >= num_frames:
        indices = np.linspace(0, total - 1, num_frames, dtype=int)
    else:
        # Repeat frames if video is too short
        indices = np.linspace(0, total - 1, num_frames, dtype=int)
    
    sampled = [frames[i] for i in indices]
    
    # Resize and convert BGR -> RGB
    resized = []
    for f in sampled:
        f_resized = cv2.resize(f, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        f_rgb = cv2.cvtColor(f_resized, cv2.COLOR_BGR2RGB)
        resized.append(f_rgb)
    
    return np.stack(resized, axis=0).astype(np.uint8)  # (T, H, W, 3)


def process_one_video(row):
    """Process a single video and save as NPZ. Returns metadata dict or None."""
    video_id = row["video_id"]
    video_path = row["video_path"]
    gloss = row["gloss"]
    label = row["label"]
    split = row["split"]
    
    save_path = os.path.join(PREPROCESSED_DIR, split, f"{video_id}.npz")
    
    # Skip if already exists
    if os.path.exists(save_path):
        return {
            "video_id": video_id,
            "gloss": gloss,
            "label": label,
            "split": split,
            "save_path": save_path,
            "status": "skipped"
        }
    
    # Check if source video exists
    if not os.path.exists(video_path):
        return None
    
    # Preprocess
    frames = preprocess_video(video_path, NUM_FRAMES, IMAGE_SIZE)
    if frames is None:
        return None
    
    # Save as compressed NPZ (uint8)
    np.savez_compressed(save_path, frames=frames)
    
    return {
        "video_id": video_id,
        "gloss": gloss,
        "label": label,
        "split": split,
        "save_path": save_path,
        "status": "processed"
    }


# Load train and val manifests
train_df = pd.read_csv(os.path.join(MANIFESTS_DIR, "demo_train_manifest.csv"))
val_df = pd.read_csv(os.path.join(MANIFESTS_DIR, "demo_val_manifest.csv"))

all_rows = pd.concat([train_df, val_df], ignore_index=True).to_dict("records")
print(f"📊 Total videos to preprocess: {len(all_rows)}")

# Check how many already exist
existing_train = len(list(Path(os.path.join(PREPROCESSED_DIR, "train")).glob("*.npz")))
existing_val = len(list(Path(os.path.join(PREPROCESSED_DIR, "val")).glob("*.npz")))
print(f"   Already preprocessed: train={existing_train}, val={existing_val}")

# Process with 4 workers
NUM_WORKERS = 4
results = []
failed = 0

print(f"\n🔄 Preprocessing with {NUM_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_one_video, row): row for row in all_rows}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Preprocessing"):
        result = future.result()
        if result is not None:
            results.append(result)
        else:
            failed += 1

# Summary
processed = sum(1 for r in results if r["status"] == "processed")
skipped = sum(1 for r in results if r["status"] == "skipped")

print(f"\n✅ Preprocessing complete!")
print(f"   Processed: {processed}")
print(f"   Skipped (already existed): {skipped}")
print(f"   Failed: {failed}")

# Save preprocessed manifest
df_preprocessed = pd.DataFrame(results)
preprocessed_manifest_path = os.path.join(MANIFESTS_DIR, "demo_preprocessed_manifest.csv")
df_preprocessed.to_csv(preprocessed_manifest_path, index=False)
print(f"\n💾 Saved preprocessed manifest to: {preprocessed_manifest_path}")

# Check disk usage
total_size = 0
for split in ["train", "val"]:
    split_dir = os.path.join(PREPROCESSED_DIR, split)
    for npz_file in Path(split_dir).glob("*.npz"):
        total_size += npz_file.stat().st_size

print(f"\n💾 Disk space used: {total_size / (1024**3):.2f} GB")
print(f"   Average per video: {total_size / max(len(results), 1) / (1024**2):.2f} MB")

In [ ]:
# ---------- Cell 14: Check Class Distribution After Preprocessing ----------

# Load the successfully preprocessed manifest
df_preprocessed = pd.read_csv(os.path.join(MANIFESTS_DIR, "demo_preprocessed_manifest.csv"))

print(f"✅ Total preprocessed videos: {len(df_preprocessed)}")
print(f"   Train: {len(df_preprocessed[df_preprocessed['split'] == 'train'])}")
print(f"   Val: {len(df_preprocessed[df_preprocessed['split'] == 'val'])}")

# Check class distribution
class_counts = df_preprocessed.groupby("gloss").size().sort_values()

print(f"\n📊 Class distribution ({len(class_counts)} classes):")
print(f"   Min samples: {class_counts.min()} ({class_counts.idxmin()})")
print(f"   Max samples: {class_counts.max()} ({class_counts.idxmax()})")
print(f"   Mean samples: {class_counts.mean():.1f}")
print(f"   Median samples: {class_counts.median():.1f}")

# Show classes with < 20 samples (may need extra augmentation)
sparse_classes = class_counts[class_counts < 20]
if len(sparse_classes) > 0:
    print(f"\n⚠️ Sparse classes (< 20 samples): {len(sparse_classes)}")
    for gloss, count in sparse_classes.items():
        print(f"   {gloss}: {count}")
else:
    print(f"\n✅ All classes have >= 20 samples")

# Show bottom 10 and top 10
print(f"\n📋 Bottom 10 classes:")
for gloss, count in class_counts.head(10).items():
    print(f"   {gloss:12s}: {count}")

print(f"\n📋 Top 10 classes:")
for gloss, count in class_counts.tail(10).items():
    print(f"   {gloss:12s}: {count}")

## ⚠️ SWITCH TO GPU FOR TRAINING

Preprocessing is complete. Now:

1. **Go to Settings → Accelerator → GPU**
2. **Restart the notebook** (Run → Restart & Clear Outputs)
3. **Re-run cells 1-11** (they're fast, just loading data and building model)
4. **Skip cells 12-14** (preprocessing is done, data is saved)
5. **Continue from Cell 16** (training loop)

The preprocessed `.npz` files are saved in `/kaggle/working/SignBridge_demo/preprocessed/` and will persist.

**Current version: v3 (55 classes)** - includes he, they pronouns.

In [ ]:
# ---------- Cell 16: Dataset & DataLoaders for Fine-Tuning ----------

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

class PreprocessedVideoDataset(Dataset):
    """Dataset for loading preprocessed NPZ video files with augmentation."""
    
    def __init__(self, manifest_df, augment=False):
        self.records = manifest_df.to_dict("records")
        self.augment = augment
    
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, idx):
        record = self.records[idx]
        npz_path = record["save_path"]
        label = record["label"]
        
        # Load preprocessed frames
        data = np.load(npz_path)
        frames = data["frames"]  # (T, H, W, 3), uint8
        
        # Convert to float [0, 1]
        frames = frames.astype(np.float32) / 255.0
        
        # Apply augmentations (training only)
        if self.augment:
            frames = self._augment(frames)
        
        # Normalize to [-1, 1] (matching I3D training)
        frames = (frames - 0.5) * 2.0
        
        # Convert to (C, T, H, W) for I3D
        frames = np.transpose(frames, (3, 0, 1, 2))  # (3, T, H, W)
        
        return torch.from_numpy(frames), label
    
    def _augment(self, frames):
        """Apply augmentations matching our 87.6% training."""
        T, H, W, C = frames.shape
        
        # Horizontal flip (50%)
        if random.random() < 0.5:
            frames = frames[:, :, ::-1, :].copy()
        
        # Temporal jitter: random crop of 28-32 frames, then resize back to 32
        if random.random() < 0.5 and T >= 28:
            crop_len = random.randint(28, min(T, 32))
            start = random.randint(0, T - crop_len)
            frames = frames[start:start+crop_len]
            # Resize back to 32 frames
            if len(frames) != 32:
                indices = np.linspace(0, len(frames)-1, 32, dtype=int)
                frames = frames[indices]
        
        # Brightness/contrast jitter
        if random.random() < 0.3:
            brightness = random.uniform(0.8, 1.2)
            contrast = random.uniform(0.8, 1.2)
            frames = frames * contrast + (brightness - 1.0)
            frames = np.clip(frames, 0, 1)
        
        # Spatial crop (random 80-100% crop, resize back)
        if random.random() < 0.3:
            scale = random.uniform(0.8, 1.0)
            new_size = int(H * scale)
            y = random.randint(0, H - new_size)
            x = random.randint(0, W - new_size)
            frames = frames[:, y:y+new_size, x:x+new_size, :]
            # Resize back (using simple nearest for speed)
            import cv2
            resized = []
            for f in frames:
                resized.append(cv2.resize(f, (W, H), interpolation=cv2.INTER_LINEAR))
            frames = np.stack(resized)
        
        # Gaussian noise
        if random.random() < 0.2:
            noise = np.random.normal(0, 0.02, frames.shape).astype(np.float32)
            frames = np.clip(frames + noise, 0, 1)
        
        return frames


# Load preprocessed manifest
df_preprocessed = pd.read_csv(os.path.join(MANIFESTS_DIR, "demo_preprocessed_manifest.csv"))

train_df = df_preprocessed[df_preprocessed["split"] == "train"]
val_df = df_preprocessed[df_preprocessed["split"] == "val"]

print(f"📊 Train samples: {len(train_df)}, Val samples: {len(val_df)}")

# Create datasets
train_dataset = PreprocessedVideoDataset(train_df, augment=True)
val_dataset = PreprocessedVideoDataset(val_df, augment=False)

# DataLoaders
BATCH_SIZE = 8  # Adjust based on GPU memory (8 works for T4)
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"✅ Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Compute class weights for imbalanced classes (using dynamic NUM_CLASSES_DEMO)
class_counts = train_df.groupby("label").size()
total_samples = len(train_df)

# Inverse frequency weighting with smoothing
class_weights = torch.zeros(NUM_CLASSES_DEMO)
for label, count in class_counts.items():
    class_weights[label] = total_samples / (NUM_CLASSES_DEMO * count)

# Cap extreme weights (max 10x)
class_weights = torch.clamp(class_weights, max=10.0)
print(f"\n📊 Class weights (top 5 highest):")
top_weights = torch.topk(class_weights, 5)
for idx, weight in zip(top_weights.indices.tolist(), top_weights.values.tolist()):
    gloss = label_to_gloss_demo[idx]
    print(f"   {gloss}: {weight:.2f}x")

print(f"\n✅ Dataset & DataLoaders ready for {NUM_CLASSES_DEMO} classes!")

In [ ]:
# ---------- Cell 17: Training Loop with Fine-Tuning Strategy ----------

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import time

# ============================================================================
# Training Configuration
# ============================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device: {DEVICE}")

# Fine-tuning hyperparameters
EPOCHS = 30
LR = 1e-5  # Low LR for fine-tuning (was 1e-4 for scratch)
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
PATIENCE = 7  # Early stopping patience

# Move model to device
model_demo = model_demo.to(DEVICE)
class_weights = class_weights.to(DEVICE)

# Loss function with label smoothing and class weights
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

# Optimizer - use lower LR for backbone, higher for new classifier
backbone_params = [p for n, p in model_demo.named_parameters() if not n.startswith("logits")]
classifier_params = [p for n, p in model_demo.named_parameters() if n.startswith("logits")]

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": LR},
    {"params": classifier_params, "lr": LR * 10}  # 10x LR for new classifier head
], weight_decay=WEIGHT_DECAY)

# Cosine annealing scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

print(f"\n📋 Training config:")
print(f"   Model: {NUM_CLASSES_DEMO} classes")
print(f"   Epochs: {EPOCHS}")
print(f"   Backbone LR: {LR}, Classifier LR: {LR * 10}")
print(f"   Label smoothing: {LABEL_SMOOTHING}")
print(f"   Early stopping patience: {PATIENCE}")

# ============================================================================
# Training Functions
# ============================================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training", leave=False)
    for frames, labels in pbar:
        frames = frames.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * frames.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{100*correct/total:.1f}%"})
    
    return total_loss / total, 100 * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for frames, labels in tqdm(loader, desc="Validation", leave=False):
            frames = frames.to(device)
            labels = labels.to(device)
            
            outputs = model(frames)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * frames.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, 100 * correct / total


# ============================================================================
# Training Loop
# ============================================================================

best_val_acc = 0.0
patience_counter = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

CHECKPOINT_DIR = os.path.join(BASE_OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"\n🚀 Starting training...")
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    
    # Train
    train_loss, train_acc = train_epoch(model_demo, train_loader, criterion, optimizer, DEVICE)
    
    # Validate
    val_loss, val_acc = validate(model_demo, val_loader, criterion, DEVICE)
    
    # Update scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Record history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    epoch_time = time.time() - epoch_start
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train: {train_acc:.1f}% (loss={train_loss:.4f}) | "
          f"Val: {val_acc:.1f}% (loss={val_loss:.4f}) | "
          f"LR: {current_lr:.2e} | Time: {epoch_time:.0f}s")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        
        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model_demo.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "history": history,
            "num_classes": NUM_CLASSES_DEMO,
            "label_mapping": label_mapping_demo
        }
        
        best_path = os.path.join(CHECKPOINT_DIR, "best_demo_model.pth")
        torch.save(checkpoint, best_path)
        print(f"   💾 New best model saved! Val acc: {val_acc:.2f}%")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹️ Early stopping triggered at epoch {epoch+1}")
            break

# Save final model
final_path = os.path.join(CHECKPOINT_DIR, "final_demo_model.pth")
torch.save({
    "epoch": epoch + 1,
    "model_state_dict": model_demo.state_dict(),
    "val_acc": val_acc,
    "num_classes": NUM_CLASSES_DEMO,
    "label_mapping": label_mapping_demo
}, final_path)

total_time = time.time() - start_time
print(f"\n✅ Training complete!")
print(f"   Total time: {total_time/60:.1f} minutes")
print(f"   Best validation accuracy: {best_val_acc:.2f}%")
print(f"   Best model saved to: {best_path}")

In [ ]:
# ---------- Cell 18: Training Curves & Summary ----------

import matplotlib.pyplot as plt

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(history["val_loss"], label="Val Loss", marker="s")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history["train_acc"], label="Train Acc", marker="o")
axes[1].plot(history["val_acc"], label="Val Acc", marker="s")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Training & Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=best_val_acc, color="r", linestyle="--", alpha=0.5, label=f"Best: {best_val_acc:.1f}%")

plt.tight_layout()
plt.savefig(os.path.join(BASE_OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

# Summary
print("\n" + "="*60)
print("📊 TRAINING SUMMARY")
print("="*60)
print(f"   Model: InceptionI3d ({NUM_CLASSES_DEMO} classes)")
print(f"   Training samples: {len(train_df)}")
print(f"   Validation samples: {len(val_df)}")
print(f"   Best validation accuracy: {best_val_acc:.2f}%")
print(f"   Epochs trained: {len(history['train_loss'])}")
print(f"   Checkpoint saved: {best_path}")
print("="*60)

# Copy best model to easy download location
import shutil
final_model_path = f"/kaggle/working/demo_model_{NUM_CLASSES_DEMO}class.pth"
shutil.copy(best_path, final_model_path)
print(f"\n💾 Model copied to: {final_model_path} (ready for download)")

In [ ]:
# ---------- Cell 19: Per-Class Accuracy Analysis ----------

from collections import defaultdict

# Load best checkpoint
best_ckpt = torch.load(best_path, map_location=DEVICE)
model_demo.load_state_dict(best_ckpt["model_state_dict"])
print(f"📦 Loaded best model (epoch {best_ckpt['epoch']}, val acc {best_ckpt['val_acc']:.2f}%)")

# Evaluate per-class accuracy
model_demo.eval()
class_correct = defaultdict(int)
class_total = defaultdict(int)

with torch.no_grad():
    for frames, labels in tqdm(val_loader, desc="Evaluating"):
        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)
        
        outputs = model_demo(frames)
        _, predicted = outputs.max(1)
        
        for pred, label in zip(predicted.cpu().numpy(), labels.cpu().numpy()):
            gloss = label_to_gloss_demo[label]
            class_total[gloss] += 1
            if pred == label:
                class_correct[gloss] += 1

# Calculate per-class accuracy
per_class_acc = {}
for gloss in class_total:
    acc = 100 * class_correct[gloss] / class_total[gloss]
    per_class_acc[gloss] = (acc, class_total[gloss])

# Separate OLD (from 100-class) vs NEW classes
old_classes = {g: v for g, v in per_class_acc.items() if g in CURRENT_100_LABELS}
new_classes = {g: v for g, v in per_class_acc.items() if g not in CURRENT_100_LABELS}

print("="*60)
print("📊 PER-CLASS ACCURACY ANALYSIS")
print("="*60)

print(f"\n🔵 OLD CLASSES (from 100-class model): {len(old_classes)} classes")
old_avg = sum(v[0] for v in old_classes.values()) / len(old_classes) if old_classes else 0
print(f"   Average accuracy: {old_avg:.1f}%")
print(f"\n   {'Gloss':<12} {'Acc':>5}  {'Samples':>7}")
print(f"   {'-'*30}")
for gloss, (acc, count) in sorted(old_classes.items(), key=lambda x: x[1][0]):
    marker = "⚠️" if acc < 70 else ""
    print(f"   {gloss:<12} {acc:>5.1f}%  {count:>6}   {marker}")

print(f"\n🆕 NEW CLASSES (learned from scratch): {len(new_classes)} classes")
new_avg = sum(v[0] for v in new_classes.values()) / len(new_classes) if new_classes else 0
print(f"   Average accuracy: {new_avg:.1f}%")
print(f"\n   {'Gloss':<12} {'Acc':>5}  {'Samples':>7}")
print(f"   {'-'*30}")
for gloss, (acc, count) in sorted(new_classes.items(), key=lambda x: x[1][0]):
    marker = "⚠️" if acc < 70 else ""
    print(f"   {gloss:<12} {acc:>5.1f}%  {count:>6}   {marker}")

print("\n" + "="*60)
print("📈 SUMMARY")
print("="*60)
print(f"   OLD classes ({len(old_classes)}): {old_avg:.1f}% average")
print(f"   NEW classes ({len(new_classes)}): {new_avg:.1f}% average")
print(f"   Gap: {abs(old_avg - new_avg):.1f}%")
print(f"\n   Overall validation accuracy: {best_ckpt['val_acc']:.2f}%")
print(f"   Total classes: {NUM_CLASSES_DEMO}")

In [ ]:
# ---------- Cell 20: Test Model on Video File ----------

import cv2
import numpy as np
import torch

def preprocess_video_for_inference(video_path, num_frames=32, image_size=224):
    """Preprocess a video file for model inference."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    
    if len(frames) == 0:
        raise ValueError("Video has no frames")
    
    # Sample frames uniformly
    total = len(frames)
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    sampled = [frames[i] for i in indices]
    
    # Resize and convert BGR -> RGB
    processed = []
    for f in sampled:
        f_resized = cv2.resize(f, (image_size, image_size))
        f_rgb = cv2.cvtColor(f_resized, cv2.COLOR_BGR2RGB)
        processed.append(f_rgb)
    
    # Stack and normalize
    frames_np = np.stack(processed).astype(np.float32) / 255.0
    frames_np = (frames_np - 0.5) * 2.0  # [-1, 1]
    frames_np = np.transpose(frames_np, (3, 0, 1, 2))  # (C, T, H, W)
    
    return torch.from_numpy(frames_np).unsqueeze(0)  # (1, C, T, H, W)


def predict_video(model, video_path, label_to_gloss, device, top_k=5):
    """Run inference on a video file."""
    # Preprocess
    video_tensor = preprocess_video_for_inference(video_path)
    video_tensor = video_tensor.to(device)
    
    # Inference
    model.eval()
    with torch.no_grad():
        outputs = model(video_tensor)
        probs = torch.softmax(outputs, dim=1)
        top_probs, top_indices = torch.topk(probs, top_k)
    
    # Format results
    results = []
    for prob, idx in zip(top_probs[0].cpu().numpy(), top_indices[0].cpu().numpy()):
        gloss = label_to_gloss[int(idx)]
        results.append((gloss, float(prob)))
    
    return results


# ============================================================================
# Test on a specific video file
# ============================================================================

# Option 1: Test on a video from the validation set
print("🎬 Testing on a random validation video...")

val_sample = val_df.sample(1).iloc[0]
test_video_path = val_sample["video_path"].replace("/kaggle/working/SignBridge_demo/preprocessed/val/", "")
test_video_path = test_video_path.replace(".npz", ".mp4")

# Try to find the original video
possible_paths = [
    f"/kaggle/input/wlasl-processed/videos/{val_sample['video_id']}.mp4",
    f"/kaggle/input/asl-citizen/ASL_Citizen/videos/{val_sample['video_id']}.mp4",
]

original_video = None
for p in possible_paths:
    if os.path.exists(p):
        original_video = p
        break

if original_video:
    print(f"   Video: {original_video}")
    print(f"   True label: {val_sample['gloss']}")
    
    predictions = predict_video(model_demo, original_video, label_to_gloss_demo, DEVICE)
    
    print(f"\n📊 Top-5 Predictions:")
    for i, (gloss, prob) in enumerate(predictions, 1):
        marker = "✅" if gloss == val_sample['gloss'] else "  "
        print(f"   {marker} {i}. {gloss:12s} ({prob*100:.1f}%)")
else:
    print("   ⚠️ Could not find original video file")

# ============================================================================
# Option 2: Test on a custom video path
# ============================================================================

print("\n" + "="*60)
print("📤 TO TEST YOUR OWN VIDEO:")
print("="*60)
print("""
1. Upload your video to Kaggle:
   - Click 'Add Data' → 'Upload' → select your .mp4 file
   - Or use: from google.colab import files; files.upload()

2. Then run this code with your video path:

   VIDEO_PATH = "/kaggle/input/your-dataset/your_video.mp4"
   predictions = predict_video(model_demo, VIDEO_PATH, label_to_gloss_demo, DEVICE)
   
   print("Predictions:")
   for gloss, prob in predictions:
       print(f"  {gloss}: {prob*100:.1f}%")
""")

# ============================================================================
# Quick test function for easy reuse
# ============================================================================

def test_video(video_path):
    """Quick function to test any video."""
    if not os.path.exists(video_path):
        print(f"❌ File not found: {video_path}")
        return
    
    print(f"🎬 Testing: {video_path}")
    predictions = predict_video(model_demo, video_path, label_to_gloss_demo, DEVICE)
    
    print(f"\n📊 Predictions:")
    for i, (gloss, prob) in enumerate(predictions, 1):
        conf = "🟢" if prob > 0.7 else "🟡" if prob > 0.3 else "🔴"
        print(f"   {conf} {i}. {gloss:12s} ({prob*100:.1f}%)")
    
    return predictions

print("\n💡 Quick test: test_video('/path/to/your/video.mp4')")